In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys
from datetime import datetime
cwd = os.getcwd()
project_root_idx = cwd.split('/').index('ecom_app')
project_root_dir = '/'.join(cwd.split('/')[:project_root_idx+1])
sys.path.append(project_root_dir)


In [3]:
from pyspark.sql import functions as F

In [18]:
from ETL.python.utils.spark_utils import get_spark_session, read_table, write_table, overwrite_table

In [6]:
spark = get_spark_session()

In [7]:
## Link ETL log json here
last_run_timestamp = '2026-05-18 12:00:00'


In [8]:
# Initializations 
curr_run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [9]:
customer_table = read_table('ecom_oltp_db', 'customers')
address_table = read_table('ecom_oltp_db', 'address')
dim_customer_table = read_table('ecom_olap_db', 'dim_customer')

# Implementing SCD 2 for customer table

In [10]:
def get_updated_rows_df(df, last_run_timestamp, curr_run_timestamp, updated_at='updated_at'):
    last_run_timstamp_col = F.to_timestamp(F.lit(last_run_timestamp), 'yyyy-MM-dd HH:mm:ss')
    curr_run_timstamp_col = F.to_timestamp(F.lit(curr_run_timestamp), 'yyyy-MM-dd HH:mm:ss')
    return df.filter( (F.col(updated_at) >= last_run_timstamp_col ) & (F.col(updated_at) < curr_run_timstamp_col) )

def get_added_rows_df(df, last_run_timestamp, curr_run_timestamp, created_at='created_at'):
    last_run_timstamp_col = F.to_timestamp(F.lit(last_run_timestamp), 'yyyy-MM-dd HH:mm:ss')
    curr_run_timstamp_col = F.to_timestamp(F.lit(curr_run_timestamp), 'yyyy-MM-dd HH:mm:ss')
    return df.filter( (F.col(created_at) >= last_run_timstamp_col ) & (F.col(created_at) < curr_run_timstamp_col) )

In [11]:
updated_rows_df = get_updated_rows_df(customer_table, last_run_timestamp, curr_run_timestamp)
added_rows_df = get_added_rows_df(customer_table, last_run_timestamp, curr_run_timestamp)

In [12]:
dim_customer_table = dim_customer_table.join(updated_rows_df[['id']], on=[dim_customer_table.customer_id == updated_rows_df.id], how='left')

In [13]:
dim_customer_table = dim_customer_table.withColumn('valid_to', F.when( ((F.col('is_current') == True) & (F.col('id').isNotNull())), F.to_timestamp(F.lit(curr_run_timestamp), 'yyyy-MM-dd HH:mm:ss'))
                                            .otherwise(F.col('valid_to')))\
                                        .withColumn('is_current', F.when( ((F.col('is_current') == True) & (F.col('id').isNotNull())), F.lit(False))
                                            .otherwise(F.col('is_current')))\
                                        .drop('id')

In [14]:
append_dim_cust_table = updated_rows_df.unionByName(added_rows_df)

In [15]:
append_dim_cust_table = append_dim_cust_table.join(address_table, on=[append_dim_cust_table.address_id == address_table.id], how='left').drop(*[address_table.id,address_table.created_at, address_table.updated_at])

In [16]:
append_dim_cust_table = append_dim_cust_table.withColumnRenamed('id', 'customer_id')\
                        .withColumn('address',    F.concat_ws(', ', *['house_number', 'street']))\
                        .withColumn('valid_from', F.to_timestamp(F.lit(curr_run_timestamp), 'yyyy-MM-dd HH:mm:ss'))\
                        .withColumn('valid_to',   F.to_timestamp(F.lit('9999-12-31 23:59:59'), 'yyyy-MM-dd HH:mm:ss'))\
                        .withColumn('is_current', F.lit(True))


In [17]:
dim_customer_table = append_dim_cust_table[dim_customer_table.columns].unionByName(dim_customer_table)

In [25]:
dim_customer_table.count()

4

26/05/20 00:46:45 ERROR RetryingBlockTransferor: Exception while beginning fetch of 1 outstanding blocks
java.io.IOException: Failed to connect to /172.21.0.5:41047
	at org.apache.spark.network.client.TransportClientFactory.createClient(TransportClientFactory.java:304)
	at org.apache.spark.network.client.TransportClientFactory.createClient(TransportClientFactory.java:224)
	at org.apache.spark.network.netty.NettyBlockTransferService$$anon$2.createAndStart(NettyBlockTransferService.scala:137)
	at org.apache.spark.network.shuffle.RetryingBlockTransferor.transferAllOutstanding(RetryingBlockTransferor.java:180)
	at org.apache.spark.network.shuffle.RetryingBlockTransferor.start(RetryingBlockTransferor.java:159)
	at org.apache.spark.network.netty.NettyBlockTransferService.fetchBlocks(NettyBlockTransferService.scala:157)
	at org.apache.spark.network.BlockTransferService.fetchBlockSync(BlockTransferService.scala:102)
	at org.apache.spark.storage.BlockManager.fetchRemoteManagedBuffer(BlockManage

In [ ]:
dim_customer_table.show()

+-----------+--------+-------------+--------------------+-----------------+---------+-----------+-------+------+-------------------+-------------------+----------+-------------------+
|customer_id|    name|mobile_number|               email|          address|     city|   locality|pincode|status|         valid_from|           valid_to|is_current|         created_at|
+-----------+--------+-------------+--------------------+-----------------+---------+-----------+-------+------+-------------------+-------------------+----------+-------------------+
|      20001|  anusha|   7894561230|       abc@gmail.com|      20, MG Road|   Mumbai|    Andheri| 475919|active|2026-05-20 00:39:28|9999-12-31 23:59:59|      true|2026-05-19 23:56:47|
|      20001|  anusha|   7894561230|       abc@gmail.com|      20, MG Road|   Mumbai|    Andheri| 475919|active|2026-05-20 00:39:28|9999-12-31 23:59:59|      true|2026-05-19 23:56:47|
|      16896|abhishek|   8946281833|customer16896@exa...|967, Brigade Road|Banga

26/05/20 00:45:29 ERROR RetryingBlockTransferor: Exception while beginning fetch of 1 outstanding blocks (after 3 retries)
java.io.IOException: Connecting to /172.21.0.4:44547 failed in the last 4750 ms, fail this connection directly
	at org.apache.spark.network.client.TransportClientFactory.createClient(TransportClientFactory.java:220)
	at org.apache.spark.network.netty.NettyBlockTransferService$$anon$2.createAndStart(NettyBlockTransferService.scala:137)
	at org.apache.spark.network.shuffle.RetryingBlockTransferor.transferAllOutstanding(RetryingBlockTransferor.java:180)
	at org.apache.spark.network.shuffle.RetryingBlockTransferor.lambda$initiateRetry$0(RetryingBlockTransferor.java:227)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1136)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:635)
	at io.netty.util.concurrent.FastThreadLocalRunnable.run(FastThreadLocalRunnable.java:30)
	at java.base/java.lang.Th

In [ ]:
overwrite_table('ecom_olap_db', 'dim_customer', dim_customer_table)

truncate successful


26/05/20 00:44:03 WARN TaskSetManager: Lost task 0.0 in stage 12.0 (TID 12) (172.21.0.4 executor 1): java.sql.BatchUpdateException: Duplicate entry '20001' for key 'dim_customer.PRIMARY'
	at com.mysql.cj.jdbc.exceptions.SQLError.createBatchUpdateException(SQLError.java:223)
	at com.mysql.cj.jdbc.ClientPreparedStatement.executeBatchSerially(ClientPreparedStatement.java:813)
	at com.mysql.cj.jdbc.ClientPreparedStatement.executeBatchInternal(ClientPreparedStatement.java:416)
	at com.mysql.cj.jdbc.StatementImpl.executeBatch(StatementImpl.java:802)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.savePartition(JdbcUtils.scala:865)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.$anonfun$saveTable$1(JdbcUtils.scala:1018)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.$anonfun$saveTable$1$adapted(JdbcUtils.scala:1017)
	at org.apache.spark.rdd.RDD.$anonfun$foreachPartition$2(RDD.scala:1047)
	at org.apache.spark.rdd.RDD.$anonfun$foreachPartition$2$ad

Log:  An error occurred while calling o227.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 12.0 failed 4 times, most recent failure: Lost task 0.3 in stage 12.0 (TID 16) (172.21.0.4 executor 1): java.sql.BatchUpdateException: Duplicate entry '20001' for key 'dim_customer.PRIMARY'
	at com.mysql.cj.jdbc.exceptions.SQLError.createBatchUpdateException(SQLError.java:223)
	at com.mysql.cj.jdbc.ClientPreparedStatement.executeBatchSerially(ClientPreparedStatement.java:813)
	at com.mysql.cj.jdbc.ClientPreparedStatement.executeBatchInternal(ClientPreparedStatement.java:416)
	at com.mysql.cj.jdbc.StatementImpl.executeBatch(StatementImpl.java:802)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.savePartition(JdbcUtils.scala:865)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.$anonfun$saveTable$1(JdbcUtils.scala:1018)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.$anonfun$saveTable$1$adapted(JdbcUtils.scala:1

26/05/20 00:44:09 ERROR RetryingBlockTransferor: Exception while beginning fetch of 1 outstanding blocks (after 2 retries)
java.io.IOException: Connecting to /172.21.0.4:44547 failed in the last 4750 ms, fail this connection directly
	at org.apache.spark.network.client.TransportClientFactory.createClient(TransportClientFactory.java:220)
	at org.apache.spark.network.netty.NettyBlockTransferService$$anon$2.createAndStart(NettyBlockTransferService.scala:137)
	at org.apache.spark.network.shuffle.RetryingBlockTransferor.transferAllOutstanding(RetryingBlockTransferor.java:180)
	at org.apache.spark.network.shuffle.RetryingBlockTransferor.lambda$initiateRetry$0(RetryingBlockTransferor.java:227)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1136)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:635)
	at io.netty.util.concurrent.FastThreadLocalRunnable.run(FastThreadLocalRunnable.java:30)
	at java.base/java.lang.Th